In [ ]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Load dataset
df = pd.read_csv("../02_Data/Raw/ai4i2020.csv")


# Remove unnecessary columns
df.drop(columns=["UDI", "Product ID"], inplace=True)


# Encode categorical feature
encoder = LabelEncoder()
df["Type"] = encoder.fit_transform(df["Type"])


# Feature Engineering

df["Temperature Difference"] = (
    df["Process temperature [K]"] -
    df["Air temperature [K]"]
)


df["Tool Wear Level"] = pd.cut(
    df["Tool wear [min]"],
    bins=[0, 100, 200, df["Tool wear [min]"].max()],
    labels=[0, 1, 2],
    include_lowest=True
)

df["Tool Wear Level"] = df["Tool Wear Level"].astype(int)


# Separate features and target

X = df.drop("Machine failure", axis=1)
y = df["Machine failure"]


# Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Feature scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Define models

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),

    "KNN": KNeighborsClassifier()

}


# Train and evaluate models

results = []

for name, model in models.items():

    if name in ["Logistic Regression", "KNN"]:

        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)

    else:

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)


    results.append({

        "Model": name,

        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": precision_score(
            y_test,
            predictions
        ),

        "Recall": recall_score(
            y_test,
            predictions
        ),

        "F1 Score": f1_score(
            y_test,
            predictions
        )

    })


results_df = pd.DataFrame(results)

display(results_df)

# Industrial Equipment Reliability Intelligence Platform

## Model Training

This notebook trains multiple machine learning classification models to predict machine failures. Class imbalance is handled using class weighting to give more importance to failure cases during model training.